In [1]:
# clone repo

import os

PROJECT_ROOT = "/content/Project_Generative_AI_for_Data_Augmentation"

if not os.path.exists(PROJECT_ROOT):
    !git clone https://github.com/Jorj91/Project_Generative_AI_for_Data_Augmentation.git {PROJECT_ROOT}

%cd {PROJECT_ROOT}

Cloning into '/content/Project_Generative_AI_for_Data_Augmentation'...
remote: Enumerating objects: 415, done.
remote: Counting objects: 100% (225/225), done.
remote: Compressing objects: 100% (173/173), done.
remote: Total 415 (delta 121), reused 116 (delta 52), pack-reused 190 (from 1)
Receiving objects: 100% (415/415), 16.41 MiB | 36.07 MiB/s, done.
Resolving deltas: 100% (212/212), done.
/content/Project_Generative_AI_for_Data_Augmentation


In [ ]:
# TO DO: RUN FOR WHOLE SCOPE

In [ ]:
# TO DO: add comments!!!

In [2]:
# CONTROLLED VERBOSITY

# Disable HF download progress bars BEFORE importing anything HF-related

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

In [3]:
# Dependency install
INSTALL_DEPS = True

if INSTALL_DEPS:
    !pip install -r requirements.txt -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.1 MB/s eta 0:00:00


In [4]:
import sys
sys.path.append(os.path.join(PROJECT_ROOT, "src"))

import importlib

import captioning
importlib.reload(captioning)
from captioning import run_captioning


import training_evaluation
importlib.reload(training_evaluation)
from training_evaluation import run_training

In [5]:
# Setup

import torch
import logging
from torchvision.datasets import OxfordIIITPet
import numpy as np
from torch.utils.data import Subset
from sklearn.model_selection import train_test_split
from transformers import Blip2Processor, Blip2ForConditionalGeneration
import random
from transformers.utils import logging as transformers_logging
from huggingface_hub.utils import logging as hf_logging


# Silence transformers & HF logs (keep only errors)
transformers_logging.set_verbosity_error()
hf_logging.set_verbosity_error()

logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

In [6]:
# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

In [7]:
import gc

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    print("GPU memory cleared.")

In [8]:
PROJECT_ROOT # it must be /content/Project_Generative_AI_for_Data_Augmentation

'/content/Project_Generative_AI_for_Data_Augmentation'

In [9]:
# control flags.
RUN_CAPTIONING = True
RUN_TEXT_VARIATION = True
RUN_IMAGE_GENERATION = True
RUN_TRAINING = True

In [10]:
# dataset loading

dataset_train = OxfordIIITPet(
    root=os.path.join(PROJECT_ROOT, "data", "raw"),
    split="trainval",
    download=True
)

dataset_test = OxfordIIITPet(
    root=os.path.join(PROJECT_ROOT, "data", "raw"),
    split="test",
    download=True
)

print("Train size:", len(dataset_train))
print("Test size:", len(dataset_test))

100%|██████████| 792M/792M [00:26<00:00, 30.2MB/s]
100%|██████████| 19.2M/19.2M [00:01<00:00, 16.0MB/s]


Train size: 3680
Test size: 3669


In [11]:
# extract labels
labels = dataset_train._labels
indices = np.arange(len(dataset_train))

# perform stratified split
train_small_idx, _ = train_test_split(
    indices,
    train_size=0.30,
    stratify=labels,
    random_state=42
)

SPLIT_DIR = os.path.join(PROJECT_ROOT, "data", "splits")
os.makedirs(SPLIT_DIR, exist_ok=True)

np.save(os.path.join(SPLIT_DIR, "train_small_indices.npy"), train_small_idx)

In [12]:
# # run for ALL
# # create subset dataset from training set
dataset_train_small = Subset(dataset_train, train_small_idx)

In [ ]:
# # run for 10
# dataset_train_small_10 = Subset(dataset_train, train_small_idx[:10])

# Captioning

In [13]:
CAPTION_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "captions",
    "captions_train_small.json"
)

In [14]:
# 13 minutes with A100 GPU
if RUN_CAPTIONING:

  device = "cuda" if torch.cuda.is_available() else "cpu"

  processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")

  model = Blip2ForConditionalGeneration.from_pretrained(
      "Salesforce/blip2-opt-2.7b",
      torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32 # to reduce GPU memory usage
  )

  model.to(device)
  model.eval()



  captions_dict = run_captioning(
      dataset_train_small=dataset_train_small, # FOR ALL
      # dataset_train_small=dataset_train_small_10, # FOR 10
      model=model,
      processor=processor,
      device=device,
      output_path=CAPTION_PATH
  )

  del model
  del processor
  clear_gpu()
  !nvidia-smi

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
100%|██████████| 1104/1104 [13:00<00:00,  1.41it/s]


Full caption generation completed.
GPU memory cleared.
Mon Feb 23 17:55:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             54W /  400W |     546MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disa

In [15]:
# check date time of last change of given file
!stat "$CAPTION_PATH"

  File: /content/Project_Generative_AI_for_Data_Augmentation/data/captions/captions_train_small.json
  Size: 210784    	Blocks: 416        IO Block: 4096   regular file
Device: 4bh/75d	Inode: 1867820     Links: 1
Access: (0644/-rw-r--r--)  Uid: (    0/    root)   Gid: (    0/    root)
Access: 2026-02-23 17:55:23.614029025 +0000
Modify: 2026-02-23 17:55:13.544065796 +0000
Change: 2026-02-23 17:55:13.544065796 +0000
 Birth: 2026-02-23 17:42:15.790667957 +0000


# Text variation

## FLAN-T5-Large Model



In [16]:
import text_variation_flan_large
importlib.reload(text_variation_flan_large)
from text_variation_flan_large import run_text_variation as run_flan_large

In [17]:
MAX_ITEMS = None # None = full dataset

In [18]:
CAPTION_PATH = os.path.join(
    PROJECT_ROOT,
    "data",
    "captions",
    "captions_train_small.json"
)

OUTPUT_PATH_FLAN_L = os.path.join(
    PROJECT_ROOT,
    "data",
    "captions",
    "captions_train_small_flan_large.json"
)

In [ ]:
run_flan_large(
    caption_file=CAPTION_PATH,
    output_file=OUTPUT_PATH_FLAN_L,
    max_items=None
)

 90%|████████▉ | 992/1104 [36:56<04:23,  2.36s/it]

In [ ]:
clear_gpu()
!nvidia-smi

In [ ]:
from google.colab import files
files.download(OUTPUT_PATH_FLAN_L)

FLAN-T5-Large was evaluated as a candidate model for caption rewriting.

However, qualitative analysis showed several consistent issues:

- **poor semantic preservation:** generated output often drifted away from original meaning

- **hallucinations:** frequent introduction of unrelated object, people, or scenes

Due to these limitations, FLAN-T5-Large was deemed unsuitable for controlled data augmentation.

## FLAN-T5-XL Model

In [ ]:
from text_variation_flan_xl import run_text_variation as run_flan_xl

OUTPUT_PATH_FLAN_XL = os.path.join(
    PROJECT_ROOT,
    "data",
    "captions",
    "captions_train_small_flan_xl.json"
)

run_flan_xl(
    caption_file=CAPTION_PATH,
    output_file=OUTPUT_PATH_FLAN_XL,
    max_items=None  # None = full dataset
)

In [ ]:
clear_gpu()
!nvidia-smi

In [ ]:
from google.colab import files
files.download(OUTPUT_PATH_FLAN_XL)

FLAN-T5-XL produced grammatically correct and semantically faithful rewrites.

However, the generated variations showed very low lexical diversity, often resulting in near-duplicate sentences with only minor wording or punctuation changes.

Since the goal of this stage is meaningful data augmentation, higher variation diversity was required.

## Mistral 7B Instruct Model

In [ ]:
from text_variation_mistral import run_text_variation as run_mistral

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
CAPTIONS_DIR = os.path.join(DATA_DIR, "captions")
TEXT_VARIATIONS_DIR = os.path.join(DATA_DIR, "text_variations")


os.makedirs(TEXT_VARIATIONS_DIR, exist_ok=True)

CAPTION_FILE = os.path.join(
    CAPTIONS_DIR,
    "captions_train_small_10.json"
)

TEXT_VARIATION_FILE = os.path.join(
    TEXT_VARIATIONS_DIR,
    "text_variations_train_small_10.json"
)

if RUN_TEXT_VARIATION:

    run_mistral(
        caption_file = CAPTION_FILE,
        output_file= TEXT_VARIATION_FILE,
        max_items=10
    )

In [ ]:
!stat $TEXT_VARIATION_FILE

In [ ]:
clear_gpu()
!nvidia-smi

In [ ]:
from google.colab import files
files.download(TEXT_VARIATION_FILE)

In [ ]:
# force runtime termination in code
import os
os.kill(os.getpid(), 9)

# Caption Selection

In [ ]:
from src.image_generation import (CaptionSelector, SyntheticImageGenerator)
import json

In [ ]:
with open(TEXT_VARIATION_FILE, "r") as f:
    text_variations = json.load(f)

In [ ]:
selector = CaptionSelector()

selected_data = {}

for idx, data in text_variations.items():

    class_name = data["class_name"]
    original_captions = data["original_captions"]
    generated_captions = data["generated_captions"]

    selected_generated = selector.select_top_captions(
        original_captions,
        generated_captions,
        top_k=2
    )

    selected_data[idx] = {
    "class_name": class_name,
    "original_captions": original_captions,
    "selected_generated_captions": selected_generated
}

In [ ]:
selected_data

# Image Generation

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())

In [ ]:
if RUN_IMAGE_GENERATION:
        generator = SyntheticImageGenerator()

        metadata = generator.generate_images(
                selected_data = selected_data,
                output_dir = os.path.join(DATA_DIR, "synthetic/images"),
                checkpoint_file = os.path.join(DATA_DIR, "synthetic/generation_checkpoint.json"),
                final_metadata_file = os.path.join(DATA_DIR, "synthetic/generation_metadata_final.json"),
                batch_size=4
        )

        del generator
        clear_gpu()
        !nvidia-smi

In [ ]:
# check date time last edit
img_file = os.path.join(DATA_DIR, "synthetic/images/1403_0.png")
!stat $img_file

In [ ]:
'''import shutil

shutil.make_archive(
    "synthetic_images",
    'zip',
    os.path.join(DATA_DIR, "synthetic/images")
)'''

In [ ]:
'''from google.colab import files
files.download("synthetic_images.zip")'''

In [ ]:
'''import os
import datetime

file_path = TEXT_VARIATION_FILE

timestamp = os.path.getmtime(file_path)
print("Last modified:",
      datetime.datetime.fromtimestamp(timestamp))'''

# Training and Evaluation

In [ ]:
if RUN_TRAINING:

    results = run_training(
        PROJECT_ROOT, epochs=5, batch_size=64
    )

    print("Baseline Accuracy:", results["baseline"]["accuracy"])
    print("Classical Augmentation Accuracy:", results["classical_only"]["accuracy"])
    print("Synthetic + Classical Augmentation Accuracy:", results["synthetic_plus_classical"]["accuracy"])

In [ ]:
# ALERT: download evaluation_results.json at next run and upload it manually to git

In [ ]:
# ALERT: DOWNLOAD AT NEXT EXECUTION AND UPLOAD IT MANUALLY TO GIT
# capture exact libraries version used in current environemnt in colab
# !pip freeze | grep -E "torch|torchvision|sentence-transformers|transformers|diffusers|accelerate|sentencepiece|scikit-learn|xformers|matplotlib|numpy|pillow|tqdm|bitsandbytes" > requirements_locked.txt

In [ ]:
# TO DO: analyze results